In [ ]:
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from results_utils import MODELS_DIR, RESULTS_DIR

runs = pd.read_csv(RESULTS_DIR / "runs.csv")
latest = runs.groupby("model").tail(1).sort_values("val_acc", ascending=False).reset_index(drop=True)
models = latest["model"].tolist()

latest[["model", "params", "epochs_trained", "best_epoch", "train_acc", "val_acc", "val_loss", "test_acc", "macro_f1"]]

In [ ]:
hp_cols = [c for c in latest.columns if c.startswith("hp_")]
latest[["model"] + hp_cols]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for name in models:
    hist = pd.read_csv(RESULTS_DIR / f"{name}_history.csv")
    axes[0].plot(hist["val_accuracy"], label=name)
    axes[1].plot(hist["val_loss"], label=name)
    axes[2].plot(hist["accuracy"] - hist["val_accuracy"], label=name)

axes[0].set_title("Val Accuracy")
axes[1].set_title("Val Loss")
axes[2].set_title("Train - Val Accuracy (overfitting gap)")
axes[2].axhline(0, color="gray", linestyle="--")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
x = np.arange(len(models))
plt.figure(figsize=(2.5 * len(models) + 2, 4))
plt.bar(x - 0.2, latest["val_acc"], width=0.4, label="Val Accuracy")
plt.bar(x + 0.2, latest["test_acc"], width=0.4, label="Test Accuracy")
for i, (v, t) in enumerate(zip(latest["val_acc"], latest["test_acc"])):
    plt.text(i - 0.2, v + 0.01, f"{v:.0%}", ha="center")
    plt.text(i + 0.2, t + 0.01, f"{t:.0%}", ha="center")
plt.xticks(x, models)
plt.ylim(0, 1.1)
plt.legend()
plt.title("Accuracy per model")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(7 * len(models), 6), squeeze=False)

for ax, name in zip(axes[0], models):
    preds = pd.read_csv(RESULTS_DIR / f"{name}_predictions.csv")
    class_names = list(preds.columns[3:])
    cm = confusion_matrix(preds["true_label"], preds["pred_label"], labels=class_names)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
    acc = latest.loc[latest["model"] == name, "test_acc"].item()
    ax.set_title(f"{name} - test accuracy {acc:.0%}")

plt.tight_layout()
plt.show()

In [ ]:
f1 = pd.DataFrame({name: pd.read_csv(RESULTS_DIR / f"{name}_report.csv", index_col=0)["f1-score"] for name in models})
f1 = f1.drop(index=["accuracy", "macro avg", "weighted avg"])

f1.plot.bar(figsize=(14, 4), width=0.8)
plt.ylim(0, 1.05)
plt.ylabel("F1-score")
plt.title("F1-score per sport")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

f1.round(2)

In [ ]:
best = latest.sort_values(["val_acc", "val_loss"], ascending=[False, True]).iloc[0]

shutil.copy(MODELS_DIR / f"{best['model']}_best.keras", MODELS_DIR / "best_model.keras")
(RESULTS_DIR / "best_model.txt").write_text(f"{best['model']}\nval_acc={best['val_acc']}\ntest_acc={best['test_acc']}\n")
latest.to_csv(RESULTS_DIR / "comparison.csv", index=False)

print(f"Best model (chosen on validation): {best['model']}")
print(f"Val accuracy: {best['val_acc']:.2%} | Test accuracy: {best['test_acc']:.2%}")